In [1]:
# === 1. Import thư viện ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

import os
from docx import Document
import re

import zipfile
import json
from lxml import etree

pd.set_option('display.max_rows', 200)   # mặc định 60
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


d:\data datn\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# finetune

In [2]:
# === 2. Đọc dữ liệu CSV đã gắn nhãn ===
df = pd.read_csv("src_data/all_labeled_questions_2.csv")  # 🔹 đường dẫn tới file bạn tải lên

# Kiểm tra dữ liệu
print(df.head())

# === 3. Ánh xạ nhãn thành số ===
label_map = {"question": 0, "answer": 1}
df["label_id"] = df["label"].map(label_map)



# === 4. Chia tập train/test ===
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

# === 5. Chuẩn bị tokenizer ===
model_name = "vinai/phobert-base"  # ⚠️ Nếu dữ liệu là tiếng Việt, nên đổi thành "bert-base-multilingual-cased" hoặc "PhoBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

# === 6. Tạo dataset cho HuggingFace ===
train_dataset = Dataset.from_pandas(train_df[['content', 'label_id']])
test_dataset = Dataset.from_pandas(test_df[['content', 'label_id']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# === 7. Load mô hình gốc ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_map))

# === 8. Cấu hình huấn luyện ===
training_args = TrainingArguments(
    output_dir="./model_output",
    do_eval=True,   # để mô hình vẫn chạy đánh giá trên eval_dataset
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # bật FP16 nếu có GPU
    report_to="none"
)

# === 9. Huấn luyện ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# === 10. Lưu mô hình đã fine-tune ===
save_path = "bert_question_answer_classifier"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Fine-tune hoàn tất. Mô hình đã lưu tại:", save_path)
print("Label map:", label_map)

      label                                                                                           content
0  question                                           Câu 1: Trong điều kiện chuẩn về nhiệt độ và áp suất thì
1    answer                   A. số phân tử trong một đơn vị thể tích của các chất khí khác nhau là như nhau.
2    answer                       B. các phân tử của các chất khí khác nhau chuyển động với vận tốc như nhau.
3    answer                        C. khoảng cách giữa các phân tử rất nhỏ so với kích thước của các phân tử.
4    answer  D. các phân tử khí khác nhau va chạm vào thành bình tác dụng vào thành bình những lực bằng nhau.


d:\data datn\env\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Map: 100%|██████████| 2060/2060 [00:00<00:00, 5716.17 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  6%|▋         | 50/771 [00:19<04:29,  2.68it/s]

{'loss': 0.2675, 'grad_norm': 0.38544511795043945, 'learning_rate': 1.872892347600519e-05, 'epoch': 0.19}


 13%|█▎        | 100/771 [00:38<04:10,  2.68it/s]

{'loss': 0.0303, 'grad_norm': 1.6562780141830444, 'learning_rate': 1.745784695201038e-05, 'epoch': 0.39}


...

{'loss': 0.0077, 'grad_norm': 0.7845101356506348, 'learning_rate': 1.91958495460441e-06, 'epoch': 2.72}


 97%|█████████▋| 750/771 [04:46<00:07,  2.68it/s]

{'loss': 0.0094, 'grad_norm': 0.016297250986099243, 'learning_rate': 6.22568093385214e-07, 'epoch': 2.91}


100%|██████████| 771/771 [04:56<00:00,  2.60it/s]


{'train_runtime': 296.4824, 'train_samples_per_second': 83.357, 'train_steps_per_second': 2.6, 'train_loss': 0.03343502433074278, 'epoch': 2.99}
✅ Fine-tune hoàn tất. Mô hình đã lưu tại: bert_question_answer_classifier
Label map: {'question': 0, 'answer': 1}


In [2]:
# Cell 2 — Import + Config + Hàm xử lý DOCX

from __future__ import annotations
import os
import re
from collections import Counter
from dataclasses import dataclass
from typing import List, Iterable, Tuple

from docx import Document


@dataclass
class Config:
    # Regex drop
    drop_patterns: Tuple[re.Pattern, ...] = (
        # Page numbers
        re.compile(r"^\s*(page\s*)?\d+\s*/\s*\d+\s*$", re.I),
        re.compile(r"^\s*(page\s*)?\d+\s*$", re.I),

        # VN headings
        re.compile(r"^\s*(mục lục|lời nói đầu|lời mở đầu|tài liệu tham khảo|phụ lục)\s*$", re.I),
        re.compile(r"^\s*(chương|phần|mục)\s+([ivxlcdm]+|\d+)\b.*$", re.I),

        # Numbered headings: 1. / 1) / 1.1 / I. / A.
        re.compile(r"^\s*((\d+(\.\d+){0,6})|([ivxlcdm]+))\s*[\.\)]\s+\S.*$", re.I),


        # Captions
        # re.compile(r"^\s*(hình|figure|fig|bảng|table)\s*\d+(\.\d+)?\s*[:\-].*$", re.I),
        # re.compile(r"^\s*(nguồn|source)\s*[:\-].*$", re.I),
    )

    # Heuristic title detection
    max_title_len: int = 80
    upper_ratio_threshold: float = 0.75
    min_words_for_title: int = 2

    # Remove repeated lines (header/footer)
    repeated_min_count: int = 3
    repeated_max_len: int = 80

    # TOC detection
    toc_scan_lines: int = 200
    toc_strong_min_lines: int = 8


CFG = Config()


def normalize_spaces(s: str) -> str:
    s = (s or "").replace("\u00a0", " ")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def mostly_upper(line: str, threshold: float) -> bool:
    letters = [ch for ch in line if ch.isalpha()]
    if not letters:
        return False
    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters) >= threshold


def looks_like_title(line: str, cfg: Config) -> bool:
    t = normalize_spaces(line)
    if not t or len(t) > cfg.max_title_len:
        return False
    words = re.findall(r"\w+", t, flags=re.UNICODE)
    if len(words) < cfg.min_words_for_title:
        return False
    return mostly_upper(t, cfg.upper_ratio_threshold)


def matches_drop(line: str, cfg: Config) -> bool:
    t = normalize_spaces(line)
    return any(pat.match(t) for pat in cfg.drop_patterns)


def toc_line_score(line: str) -> int:
    t = normalize_spaces(line)
    if not t:
        return 0
    score = 0
    if re.search(r"\.{3,}", t):          # dot leaders
        score += 2
    if re.search(r"\s\d+\s*$", t):       # ends with number
        score += 1
    if re.search(r"\b(mục|chương|phần|table of contents|contents)\b", t, re.I):
        score += 1
    return score

# --- NEW: nhận diện 1 dòng đáp án A/B/C/D ---
def is_mcq_option_line(line: str) -> bool:
    t = normalize_spaces(line)
    return bool(re.match(r"^\s*([A-Da-d])\s*([.)\:：\-])\s+.+$", t))


# --- NEW: tách A/B/C/D nếu cùng 1 dòng ---
_OPTION_MARK_RE = re.compile(r"(?i)\b([A-D])\s*([.)\:：\-])\s+")

def split_mcq_options_in_line(line: str) -> List[str]:
    t = normalize_spaces(line)
    matches = list(_OPTION_MARK_RE.finditer(t))
    if len(matches) <= 1:
        return [t] if t else [""]

    out: List[str] = []

    # Keep leading text before first option (if any)
    if matches[0].start() > 0:
        lead = t[:matches[0].start()].strip()
        if lead:
            out.append(lead)

    # Slice each option segment
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(t)
        seg = t[start:end].strip()
        if seg:
            out.append(seg)

    return out


def remove_toc_block(lines: List[str], cfg: Config) -> List[str]:
    if not lines:
        return lines

    N = min(len(lines), cfg.toc_scan_lines)
    window = lines[:N]
    strong = sum(1 for ln in window if toc_line_score(ln) >= 2)

    if strong < cfg.toc_strong_min_lines:
        return lines

    cut = 0
    for i, ln in enumerate(window):
        sc = toc_line_score(ln)
        if sc >= 2 or re.search(r"\b(mục lục|table of contents|contents)\b", ln, re.I):
            cut = i + 1
        else:
            # stop when normal text resumes
            if cut > 0 and len(normalize_spaces(ln)) > 40 and sc == 0:
                break

    return lines[cut:]


def remove_repeated_header_footer(lines: List[str], cfg: Config) -> List[str]:
    normed = [normalize_spaces(ln) for ln in lines if normalize_spaces(ln)]
    c = Counter(normed)
    repeated = {
        ln for ln, cnt in c.items()
        if cnt >= cfg.repeated_min_count and len(ln) <= cfg.repeated_max_len
    }
    return [ln for ln in lines if normalize_spaces(ln) not in repeated]


def extract_docx_paragraph_lines(docx_path: str) -> List[str]:
    doc = Document(docx_path)
    lines: List[str] = []

    for p in doc.paragraphs:
        txt = p.text or ""  # images ignored automatically
        style = (p.style.name if p.style is not None else "") or ""

        # Drop heading styles directly
        if re.search(r"\b(Heading|Title)\b", style, re.I):
            continue

        lines.append(txt)

    return lines

def clean_docx_text(docx_path: str, cfg: Config = CFG) -> str:
    lines = extract_docx_paragraph_lines(docx_path)

    lines = remove_toc_block(lines, cfg)
    lines = remove_repeated_header_footer(lines, cfg)

    out: List[str] = []
    blank = False

    for ln in lines:
        t0 = normalize_spaces(ln)

        # Tách đáp án nếu A/B/C/D cùng dòng
        pieces = split_mcq_options_in_line(t0)

        for t in pieces:
            t = normalize_spaces(t)

            if not t:
                if not blank:
                    out.append("")
                blank = True
                continue

            # ✅ Ưu tiên GIỮ đáp án A/B/C/D (không cho các rule khác xóa)
            if is_mcq_option_line(t):
                out.append(t)
                blank = False
                continue

            if matches_drop(t, cfg):
                continue

            if looks_like_title(t, cfg):
                continue

            out.append(t)
            blank = False

    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text



In [ ]:
import os
from transformers import pipeline
from docx import Document
import pandas as pd
import re

# === 1. Load mô hình PhoBERT đã fine-tune ===
model_path = "bert_question_answer_classifier"
clf = pipeline(
        "text-classification",
        model=model_path,
        tokenizer=model_path,
        device=0,        # GPU
        batch_size=32
    )

def escape_special_chars(text: str) -> str:
    return (
        text
        .replace("\\", "\\\\")   # escape backslash trước
        .replace("\n", "\\n")
        .replace("\t", "\\t")
    )


# === 2. Hàm đọc đoạn văn từ DOCX ===
# === 2. Hàm đọc đoạn văn từ DOCX (CÓ PREPROCESSING) ===
def extract_paragraphs_from_docx(docx_path):
    cleaned_text = clean_docx_text(docx_path)   # 👈 gọi preprocessing của bạn
    paragraphs = [p.strip() for p in cleaned_text.splitlines() if p.strip()]
    return paragraphs


# === 3. Hàm xử lý từng file ===
def classify_docx(docx_path):
    paragraphs = extract_paragraphs_from_docx(docx_path)
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    texts = [p for p in paragraphs if p.strip()]

    preds = clf(
        texts,
        truncation=True,
        max_length=256
    )

    classified = []
    for text, pred in zip(texts, preds):
        classified.append({
            "text": escape_special_chars(text),   # 👈 ESCAPE Ở ĐÂY
            "label": label_mapping.get(pred["label"], pred["label"])
        })

    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label == "answer":
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                text = prefix + text
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    if grouped:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        return df
    else:
        return pd.DataFrame(columns=["question", "answer"])

# === 4. Xử lý nhiều file DOCX và gộp thành 1 CSV ===
input_folder = "test_data/"  # 📂 thư mục chứa các file .docx của bạn
output_csv = "src_data/all_questions_combined_1.csv"

# lấy tất cả file .docx
docx_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".docx")]
print(f"📄 Tìm thấy {len(docx_files)} file DOCX.")

# xử lý từng file
all_dataframes = []
for path in docx_files:
    print(f"🔹 Đang xử lý: {os.path.basename(path)}")
    df = classify_docx(path)
    if not df.empty:
        df["source_file"] = os.path.basename(path)
        all_dataframes.append(df)

# gộp tất cả
if all_dataframes:
    final_df = pd.concat(all_dataframes, ignore_index=True)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Hoàn tất! Tổng {len(final_df)} câu hỏi. File lưu tại: {output_csv}")
else:
    print("⚠️ Không có dữ liệu hợp lệ nào được xuất.")


📄 Tìm thấy 21 file DOCX.
🔹 Đang xử lý: 01_TdtntTayNguyen_Ntt.docx
🔹 Đang xử lý: 03_Thpt-Victory_ThptNguyenVanCu-LongChau.docx
🔹 Đang xử lý: 07_ThptCbq_ThptTdn-PhongBaoBui.docx
🔹 Đang xử lý: 08ThptTranHungDao-ThptChuVanAn-PhamThanh.docx
🔹 Đang xử lý: 1.ThptHoangViet_NguyenMinhTri.docx
🔹 Đang xử lý: 10_TruongThptDtntN_TrangLong_ThptVietDuc-HuychungNguyen.docx
🔹 Đang xử lý: 12_Thpt_EaSup(DeNopL2)-KhongtenKhong.docx
🔹 Đang xử lý: 22_Thpt_Px_Thpt_Bd-VanDao.docx
🔹 Đang xử lý: 32_TtGdnn-GdtxM_Drak_ThptLak-ThuHa.docx
🔹 Đang xử lý: 36_Thpt-Cumgar_Thpt-PhamVanDong-LieuVoThiThuy.docx
🔹 Đang xử lý: 37_ThptDtnt-Ds_Thpt-Pbc-TruongThiHongLe.docx


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔹 Đang xử lý: 39-ThptHaiBaTrung-ThptPhanDinhPhung-DinhGiangThanh.docx
🔹 Đang xử lý: DeBangMaHoa-NguyenThiLuong.docx
🔹 Đang xử lý: GdtxTinh_NguyenThiAiNhi_Matran_De_Dapan_Tinhoc2025_DaMaHoa.docx
🔹 Đang xử lý: Sinh_Dethi_Dangthihau_Thptnguyentatthanh-Thihaudang.docx
🔹 Đang xử lý: Sinh_Dethi_Nguyenthihainhu_Ttgdnngdtxtpbmt.docx
🔹 Đang xử lý: Sinh_Dethi_ThptNguyenTatThanh.docx
🔹 Đang xử lý: ThptLeHongPhong-CamVy(1).docx
🔹 Đang xử lý: Truongth,Thcs,Thpthoangviet_Dethamkhao_Toan2025.docx
🔹 Đang xử lý: Truongthptdtntn'Tranglong_Dethamkhao_Toan2025.docx
🔹 Đang xử lý: Truongthpthongduc-Dethamkhao-Toan2025.docx
✅ Hoàn tất! Tổng 1017 câu hỏi. File lưu tại: src_data/all_questions_combined_1.csv


In [2]:
#!/usr/bin/env python3
"""
Selective extractor (strict, full): trích xuất/chuyển đổi chỉ những ảnh được chọn với
dedupe và ánh xạ canonical chính xác để mọi rId trong JSON đầu ra trỏ tới file tồn tại.

Cách dùng:
    python extract_docx_selective_strict.py path/to/file.docx

- JSON đầu ra cho mỗi paragraph chứa:
    - math: danh sách OMML string
    - images: danh sách nhóm ảnh với rIds, img_rels, sha1, saved_path, rId_to_saved, reasons
"""

import zipfile
import os
import json
import posixpath
import shutil
import subprocess
import hashlib
import re
from lxml import etree

# Namespace XML
NAMESPACES = {
    "m": "http://schemas.openxmlformats.org/officeDocument/2006/math",
    "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "v": "urn:schemas-microsoft-com:vml",
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture",
    "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
    "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
}

def _r_attr(name):
    return "{%s}%s" % (NAMESPACES["r"], name)

def _resolve_target_posix(target):
    # target in rels is often like "media/image1.wmf" -> join with "word"
    return posixpath.normpath(posixpath.join("word", target))

def _find_imagemagick_cmd():
    """
    Try to locate ImageMagick CLI. Prefer 'magick' (modern IM). If only 'convert' found,
    ensure it's from ImageMagick (on Windows, skip C:\Windows\System32\convert.exe).
    """
    for cmd in ("magick", "convert"):
        path = shutil.which(cmd)
        if not path:
            continue
        # On Windows 'convert.exe' in System32 is not ImageMagick
        if os.name == "nt" and os.path.basename(path).lower() == "convert.exe":
            parent = os.path.dirname(path).lower()
            # ensure path looks like ImageMagick install
            if "imagemagick" in path.lower() or "imagemagick" in parent:
                return cmd
            else:
                # skip the System convert.exe
                continue
        return cmd
    return None

def convert_to_png(src_path, out_dir, imagemagick_cmd=None, debug=False):
    """
    Convert ANY format (PNG/JPG/BMP/TIF/WMF/EMF/SVG...) to PNG via ImageMagick.
    Returns output PNG path on success, or None on failure.
    """
    ext = os.path.splitext(src_path)[1].lower()

    # If already PNG, return original
    if ext == ".png":
        if debug:
            print(f"[convert] already png: {src_path}")
        return src_path

    # Find ImageMagick
    if not imagemagick_cmd:
        imagemagick_cmd = _find_imagemagick_cmd()

    if not imagemagick_cmd:
        if debug:
            print("[convert] ImageMagick không tìm thấy; bỏ qua convert cho", src_path)
        return None

    # Ensure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    base = os.path.splitext(os.path.basename(src_path))[0]
    out_name = f"{base}.png"
    png_path = os.path.join(out_dir, out_name)

    # avoid name collisions
    i = 1
    while os.path.exists(png_path):
        png_path = os.path.join(out_dir, f"{base}_{i}.png")
        i += 1

    # Build command
    # For modern ImageMagick: magick "in" "out"
    # For older: convert "in" "out"
    cmd = [imagemagick_cmd, src_path, png_path]

    if debug:
        print("[convert] Chạy lệnh:", " ".join(cmd))

    try:
        # run and capture stderr/stdout for debugging
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)

        if proc.returncode != 0:
            if debug:
                try:
                    stderr = proc.stderr.decode("utf-8", errors="ignore")
                except Exception:
                    stderr = str(proc.stderr)
                print("[convert] ImageMagick trả lỗi:", stderr)
            # remove possibly created file
            if os.path.exists(png_path):
                try:
                    os.remove(png_path)
                except Exception:
                    pass
            return None

        if os.path.exists(png_path):
            if debug:
                print("[convert] Thành công:", png_path)
            return png_path
        else:
            if debug:
                print("[convert] Không thấy file output sau khi chạy:", png_path)
            return None

    except FileNotFoundError:
        if debug:
            print("[convert] ImageMagick executable không tìm thấy (FileNotFoundError).")
        return None
    except Exception as e:
        if debug:
            print("[convert] Exception khi convert:", str(e))
        return None

def _sha1_bytes(b):
    h = hashlib.sha1()
    h.update(b)
    return h.hexdigest()

def _merge_images_by_hash(images_info, canonical_by_sha1, debug=False):
    groups = {}
    order = []
    for img in images_info:
        key = img.get("sha1") or img.get("img_rel") or img.get("rId")
        if key not in groups:
            groups[key] = {
                "rIds": [],
                "img_rels": [],
                "exts": set(),
                "sizes": [],
                "sha1": img.get("sha1"),
                "alt_texts": set(),
                "selected": False,
                "saved_path": None,
                "reasons": set(),
                "rId_to_saved": {}
            }
            order.append(key)
        g = groups[key]
        if img.get("rId") and img["rId"] not in g["rIds"]:
            g["rIds"].append(img["rId"])
        if img.get("img_rel") and img["img_rel"] not in g["img_rels"]:
            g["img_rels"].append(img["img_rel"])
        if img.get("ext"):
            g["exts"].add(img["ext"])
        if img.get("size"):
            g["sizes"].append(img["size"])
        if img.get("alt"):
            g["alt_texts"].add(img["alt"])
        if img.get("selected"):
            g["selected"] = True
        if img.get("saved_path") and not g["saved_path"]:
            g["saved_path"] = img["saved_path"]
        if img.get("rId"):
            g["rId_to_saved"][img["rId"]] = img.get("saved_path")
        for reason in img.get("reasons", []):
            g["reasons"].add(reason)

    merged = []
    for key in order:
        g = groups[key]
        canonical = g["saved_path"] or canonical_by_sha1.get(g["sha1"])
        if not canonical:
            for v in g["rId_to_saved"].values():
                if v:
                    canonical = v
                    break
        rId_to_saved_norm = {rid: canonical if canonical else g["rId_to_saved"].get(rid) for rid in g["rIds"]}
        merged.append({
            "rIds": g["rIds"],
            "img_rels": g["img_rels"],
            "exts": list(g["exts"]),
            "sizes": g["sizes"],
            "sha1": g["sha1"],
            "alts": list(g["alt_texts"]),
            "selected": g["selected"],
            "saved_path": canonical,
            "rId_to_saved": rId_to_saved_norm,
            "reasons": list(g["reasons"]),
        })
    if debug:
        print(f"[merge] Đã gom {len(images_info)} -> {len(merged)} nhóm ảnh")
    return merged

QUESTION_REGEX = re.compile(r'^\s*(Câu|Cau|Question)\b\s*\d+', flags=re.IGNORECASE)
KEYWORD_IN_ALT = ("hình", "hinh", "ảnh", "anh", "figure", "fig", "image")
PROXIMITY_WINDOW_DEFAULT = 3
MIN_SIZE_BYTES_DEFAULT = 5
LARGE_SIZE_BYTES_DEFAULT = 20_000
VERY_LARGE_FACTOR = 2

def extract_docx_selective_strict(docx_path, out_folder="extracted_strict",
                                  convert_to_png_flag=False,
                                  proximity_window=PROXIMITY_WINDOW_DEFAULT,
                                  min_size_bytes=MIN_SIZE_BYTES_DEFAULT,
                                  large_size_bytes=LARGE_SIZE_BYTES_DEFAULT,
                                  debug=False):
    os.makedirs(out_folder, exist_ok=True)
    media_out = os.path.join(out_folder, "media")
    os.makedirs(media_out, exist_ok=True)
    results = []

    with zipfile.ZipFile(docx_path) as zf:
        try:
            document_xml = etree.fromstring(zf.read("word/document.xml"))
        except KeyError:
            raise FileNotFoundError("word/document.xml không tìm thấy trong .docx")

        rels_map = {}
        try:
            rels_xml = etree.fromstring(zf.read("word/_rels/document.xml.rels"))
            for rel in rels_xml.findall("rel:Relationship", NAMESPACES):
                rid = rel.get("Id") or rel.get("ID")
                target = rel.get("Target")
                target_mode = rel.get("TargetMode")
                if rid and target:
                    rels_map[rid] = {"target": target, "target_mode": target_mode}
            if debug:
                print("[info] rels_map size:", len(rels_map))
        except KeyError:
            if debug:
                print("[info] Không tìm thấy word/_rels/document.xml.rels")
            rels_map = {}

        para_nodes = document_xml.findall(".//w:p", NAMESPACES)
        question_paras = set()
        para_texts = []
        for i, p in enumerate(para_nodes):
            texts = [t.text for t in p.findall(".//w:t", NAMESPACES) if t.text]
            txt = "".join(texts).strip()
            para_texts.append(txt)
            if QUESTION_REGEX.search(txt):
                question_paras.add(i)
                if debug:
                    print(f"[info] paragraph {i} nghi là câu hỏi: {txt[:80]}")

        extracted_map = {}
        canonical_by_sha1 = {}
        imagemagick_cmd = _find_imagemagick_cmd() if convert_to_png_flag else None
        if debug and convert_to_png_flag:
            print("[info] ImageMagick command:", imagemagick_cmd)

        def _get_image_alt_from_node(node):
            cur = node
            while cur is not None:
                docpr = cur.find(".//wp:docPr", NAMESPACES)
                if docpr is not None:
                    name = docpr.get("name") or ""
                    descr = docpr.get("descr") or ""
                    return (name + " " + descr).strip()
                cNvPr = cur.find(".//pic:cNvPr", NAMESPACES)
                if cNvPr is not None:
                    name = cNvPr.get("name") or ""
                    descr = cNvPr.get("descr") or ""
                    return (name + " " + descr).strip()
                cur = cur.getparent()
            return ""

        for pi, p in enumerate(para_nodes):
            text_parts = [t.text for t in p.findall(".//w:t", NAMESPACES) if t.text]
            text = "".join(text_parts).strip()

            omml_nodes = p.findall(".//m:oMath", NAMESPACES) + p.findall(".//m:oMathPara", NAMESPACES)
            math_list = [{"omml": etree.tostring(node, encoding="unicode")} for node in omml_nodes]

            images_info = []
            seen_rids = set()

            def _process_rid(rid, xml_node):
                if not rid or rid in seen_rids:
                    return
                seen_rids.add(rid)
                info = rels_map.get(rid)
                if not info:
                    images_info.append({
                        "rId": rid,
                        "img_rel": None,
                        "selected": False,
                        "reasons": ["no_rels"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return
                if info.get("target_mode") == "External":
                    images_info.append({
                        "rId": rid,
                        "img_rel": info.get("target"),
                        "selected": False,
                        "reasons": ["external"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return

                img_rel = _resolve_target_posix(info["target"])
                try:
                    raw = zf.read(img_rel)
                except KeyError:
                    images_info.append({
                        "rId": rid,
                        "img_rel": img_rel,
                        "selected": False,
                        "reasons": ["not_in_zip"],
                        "sha1": None,
                        "saved_path": None,
                    })
                    return

                size = len(raw)
                sha1 = _sha1_bytes(raw)
                alt_text = _get_image_alt_from_node(xml_node) or ""
                alt_text_l = alt_text.lower()
                ext = os.path.splitext(info["target"])[1].lower()

                prox_range = range(max(0, pi - proximity_window), min(len(para_nodes), pi + proximity_window + 1))
                near_question = any(idx in question_paras for idx in prox_range)
                preferred_rasters = (".png", ".jpg", ".jpeg", ".bmp", ".gif", ".webp")
                vector_formats = (".wmf", ".emf", ".svg")

                selected = False
                reasons = []

                if any(k in alt_text_l for k in KEYWORD_IN_ALT):
                    selected = True
                    reasons.append("has_alt_keyword")

                if ext in vector_formats:
                    selected = True
                    reasons.append("vector_auto_select")

                if not selected and near_question:
                    if ext in preferred_rasters:
                        selected = True
                        reasons.append("near_question_and_raster")
                    elif size >= large_size_bytes:
                        selected = True
                        reasons.append(f"near_question_and_large_size_{size}")
                    else:
                        reasons.append("near_question_but_no_strong_signal")

                if not selected and size >= (large_size_bytes * VERY_LARGE_FACTOR):
                    selected = True
                    reasons.append("auto_select_very_large_global")

                saved_path_out = None
                if sha1 in canonical_by_sha1:
                    saved_path_out = canonical_by_sha1[sha1]
                    selected = True
                    reasons.append("duplicate_of_canonical")
                    extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}

                if selected:
                    if img_rel in extracted_map:
                        saved_path_out = extracted_map[img_rel]["path"]
                        reasons.append("reused_existing")
                    elif saved_path_out:
                        extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}
                    else:
                        # write raw to media_out
                        orig_name = os.path.basename(img_rel)
                        # sanitize name (remove path components)
                        base_name = os.path.splitext(orig_name)[0]
                        extn = os.path.splitext(orig_name)[1]
                        candidate = f"{base_name}{extn}"
                        out_path = os.path.join(media_out, candidate)
                        if os.path.exists(out_path):
                            i = 1
                            while os.path.exists(os.path.join(media_out, f"{base_name}_{i}{extn}")):
                                i += 1
                            out_path = os.path.join(media_out, f"{base_name}_{i}{extn}")
                        try:
                            with open(out_path, "wb") as wf:
                                wf.write(raw)
                        except Exception as e:
                            reasons.append("write_failed")
                            if debug:
                                print("[error] Không thể ghi file:", out_path, e)
                            saved_path_out = None
                        else:
                            saved_path_out = out_path
                            reasons.append("extracted_now")
                            if debug:
                                print(f"[info] Extracted {img_rel} -> {saved_path_out}")

                            # convert to PNG if requested and format is convertible
                            # expand list to include .wmf, .emf, .svg, .bmp, .tif, .tiff, etc.
                            convertible_exts = (".bmp", ".tif", ".tiff", ".wmf", ".emf", ".svg", ".pcx", ".ico")
                            if convert_to_png_flag and ext in convertible_exts:
                                png_path = convert_to_png(saved_path_out, media_out, imagemagick_cmd=imagemagick_cmd, debug=debug)
                                if png_path:
                                    # remove original if convert succeeded and ext is not png
                                    try:
                                        if saved_path_out != png_path and os.path.exists(saved_path_out):
                                            os.remove(saved_path_out)
                                    except Exception:
                                        pass
                                    saved_path_out = png_path
                                    reasons.append("converted_to_png")
                                    if debug:
                                        print(f"[info] Converted {img_rel} -> {png_path}")
                                else:
                                    reasons.append("conversion_failed")
                                    if debug:
                                        print(f"[warn] Conversion failed for {out_path}")

                            # register canonical by sha1
                            if sha1 not in canonical_by_sha1 and saved_path_out:
                                canonical_by_sha1[sha1] = saved_path_out

                            extracted_map[img_rel] = {"path": saved_path_out, "sha1": sha1}

                # if canonical exists globally, override saved_path_out
                if sha1 in canonical_by_sha1:
                    saved_path_out = canonical_by_sha1[sha1]

                images_info.append({
                    "rId": rid,
                    "img_rel": img_rel,
                    "ext": ext,
                    "size": size,
                    "sha1": sha1,
                    "alt": alt_text,
                    "selected": bool(selected),
                    "saved_path": saved_path_out,
                    "reasons": reasons
                })

            # find images in runs (drawingml blip)
            for blip in p.findall(".//a:blip", NAMESPACES):
                rid = blip.get(_r_attr("embed"))
                _process_rid(rid, blip)

            # legacy VML
            for vml in p.findall(".//v:imagedata", NAMESPACES):
                rid = vml.get(_r_attr("id")) or vml.get("id")
                _process_rid(rid, vml)

            if images_info:
                images_info = _merge_images_by_hash(images_info, canonical_by_sha1, debug=debug)

            if text or math_list or images_info:
                results.append({
                    "para_index": pi,
                    "text": text,
                    "math": math_list,
                    "images": images_info
                })

    base_name = os.path.splitext(os.path.basename(docx_path))[0]
    out_json = os.path.join(out_folder, base_name + ".selective.json")
    os.makedirs(out_folder, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as jf:
        json.dump(results, jf, ensure_ascii=False, indent=2)

    if debug:
        print("[done] Đã ghi JSON ra", out_json)
    return results

if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python extract_docx_selective_strict.py path/to/file.docx")
        sys.exit(1)

    # Example invocation. Replace with sys.argv[1] or pass via CLI.
    extract_docx_selective_strict(
        # sys.argv[1],
        r"D:\_code\DATN\data\test_data\07_ThptCbq_ThptTdn-PhongBaoBui.docx",
        # r"D:\_code\DATN\data\test_data\01_TdtntTayNguyen_Ntt.docx",
        out_folder="extracted",
        convert_to_png_flag=True,
        proximity_window=3,
        min_size_bytes=5_000,
        large_size_bytes=20_000,
        debug=True
    )


<>:44: SyntaxWarning: invalid escape sequence '\W'
<>:44: SyntaxWarning: invalid escape sequence '\W'
C:\Users\nguye\AppData\Local\Temp\ipykernel_30460\97634796.py:44: SyntaxWarning: invalid escape sequence '\W'
  """


[info] rels_map size: 276
[info] paragraph 10 nghi là câu hỏi: Câu 1. Quá trình chuyển từ thể rắn sang thể khí gọi là quá trình
[info] paragraph 26 nghi là câu hỏi: Câu 3. Biểu thức nào sau đây là đúng khi biến đổi nhiệt độ từ thang Celsius sang
[info] paragraph 29 nghi là câu hỏi: Câu 4. Một ấm đun nước bằng nhôm có có khối lượng 400 g, chứa 3 lít nước được đu
[info] paragraph 31 nghi là câu hỏi: Câu 5. Trong quá trình chất khí nhận nhiệt và sinh công thì Q và A trong biểu th
[info] paragraph 33 nghi là câu hỏi: Câu 6. Khi ấn pittông từ từ xuống để nén khí trong xilanh, thì thông số nào của 
[info] paragraph 36 nghi là câu hỏi: Câu 7. Phát biểu nào sau đây là của định luật Charles ?
[info] paragraph 41 nghi là câu hỏi: Câu 8. Cho biết khối lượng riêng của không khí ở điều kiện chuẩn (nhiệt độ 273 K
[info] paragraph 43 nghi là câu hỏi: Câu 9. Sóng điện từ là quá trình lan truyền của điện từ trường biến thiên trong 
[info] paragraph 48 nghi là câu hỏi: Câu 10. Một khung dây dẫn phẳng qu

In [2]:
# === 1. Load mô hình PhoBERT đã fine-tune ===
model_path = "./bert_question_answer_classifier"
clf = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,
    return_all_scores=False,
    device=0,  # GPU = 0, CPU = -1
    batch_size=32   # ⭐ Tối ưu GPU
)

Device set to use cuda:0
d:\python\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [4]:
data = pd.read_json("./extracted/07_ThptCbq_ThptTdn-PhongBaoBui.selective.json")  # 🔹 đường dẫn tới file JSON trích xuất

In [5]:
data.head(20)

,para_index,text,math,images
0,0,SỞ GIÁO DỤC VÀ ĐÀO TẠO ĐẮK LẮK,[],[]
1,1,TRƯỜNG THPT CAO BÁ QUÁT,[],[]
2,2,(Đề thi có 04 trang),[],[]
3,3,Đơn vị phản biện: THPT Trần Đại Nghĩa,[],[]
4,4,ĐỀ THI THỬ THPT QUỐC GIA NĂM 2025,[],[]
5,5,Môn thi: VẬT LÍ KHỐI 12,[],[]
6,6,Thời gian làm bài: 50 phút không kể thời gian phát đề,[],[]
7,8,"PHẦN I. Trắc nghiệm nhiều phương án lựa chọn. Từ câu 1 đến câu 18, mỗi câu hỏi chỉ chọn 1 phương án trả lời.",[],[]
8,10,Câu 1. Quá trình chuyển từ thể rắn sang thể khí gọi là quá trình,[],[]
9,11,A. thăng hoa.B. nóng chảyC. ngưng tụ.D. đông đặc.,[],[]


In [3]:
def classify_docx(csv_path="./extracted/07_ThptCbq_ThptTdn-PhongBaoBui.selective.json"):
    data = pd.read_json(csv_path)
    paragraphs = data['text'].tolist()
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    classified = []
    for text in paragraphs:
        try:
            pred = clf(text)[0]
            label = label_mapping.get(pred["label"], pred["label"])
            classified.append({"text": text, "label": label})
        except Exception as e:
            classified.append({"text": text, "label": "error"})
            print(f"Lỗi khi xử lý: {text[:40]} -> {e}")

    # Gom nhóm question → answer
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label == "answer":
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                text = prefix + text
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    # Trả về DataFrame
    if grouped:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        return df
    else:
        return pd.DataFrame(columns=["question", "answer"])
_res = classify_docx("./extracted/07_ThptCbq_ThptTdn-PhongBaoBui.selective.json")  # 🔹 đường dẫn tới file JSON trích xuất

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [4]:
_res

,question,answer
0,Câu 1. Quá trình chuyển từ thể rắn sang thể khí gọi là quá trình,"['A. thăng hoa.B. nóng chảyC. ngưng tụ.D. đông đặc.', 'B. (c)(c)(a)(a)(b)(b)(d)(d)Câu 2. Biển báo nào không phải là biển báo nhận diện nguồn phóng xạ?', 'C. (c)', 'D. (c)', 'E. (a)', 'F. (a)', 'G. (b)', 'H. (b)', 'I. (d)', 'J. (d)', 'A. hình (b).B. hình (c).C. hình (d).D. hình (a).']"
1,Câu 3. Biểu thức nào sau đây là đúng khi biến đổi nhiệt độ từ thang Celsius sang thang Kelvin,"['A.B.', 'C. D.']"
2,"Câu 4. Một ấm đun nước bằng nhôm có có khối lượng 400 g, chứa 3 lít nước được đun trên bếp. Khi nhận được nhiệt lượng 740 kJ thì ấm đạt đến nhiệt độ. Hỏi nhiệt độ ban đầu của ấm, biết cAl = 880 J/kg. K, .","['A. 8,15 B. 8,15 KC. 22,7D. 22,7 K']"
3,Câu 5. Trong quá trình chất khí nhận nhiệt và sinh công thì Q và A trong biểu thức phải có giá trị:,['A. và B. C. Q > 0 và A < 0D.']
4,"Câu 6. Khi ấn pittông từ từ xuống để nén khí trong xilanh, thì thông số nào của khí trong xilanh thay đổi?","['A. Nhiệt độ khí giảmB. Áp suất khí tăng', 'C. Áp suất khí giảmD. Khối lượng khí tăng']"
5,Câu 7. Phát biểu nào sau đây là của định luật Charles ?,"['A. Ở áp suất không đổi, thể tích của một khối lượng khí xác định tỉ lệ nghịch với nhiệt độ tuyệt đối của nó.', 'B. Ở áp suất không đổi, thể tích của một khối lượng khí xác định tỉ lệ thuận với nhiệt độ tuyệt đối của nó.', 'C. Ở áp suất không đổi, thể tích của một khối lượng khí xác định tăng khi nhiệt độ tuyệt đối của nó giảm.', 'D. Ở áp suất không đổi, thể tích của một khối lượng khí xác định giảm với nhiệt độ tuyệt đối của nó tăng.']"
6,"Câu 8. Cho biết khối lượng riêng của không khí ở điều kiện chuẩn (nhiệt độ 273 K, áp suất ) là 1,29 . Trong một căn phòng có thể tích khi ta tăng nhiệt độ của phòng từ 280 K ở áp suất đến nhiệt độ 300 K với áp suất thì khối lượng khí thoát ra khỏi phòng là bao nhiêu ?","['A. 0,36 kg.B. 0,29 kg.C. 0,4 kg.D. 0,25 kg.']"
7,Câu 9. Sóng điện từ là quá trình lan truyền của điện từ trường biến thiên trong không gian. Khi nói về quan hệ giữa điện trường và từ trường của điện từ trường trên thì kết luận nào sau đây là đúng?,"['A. Véctơ cường độ điện trường và cảm ứng từ cùng phương và cùng độ lớn.', 'B. Tại mỗi điểm của không gian, điện trường và từ trường luôn luôn dao động ngược pha.', 'C. Tại mỗi điểm của không gian, điện trường và từ trường luôn luôn dao động lệch pha nhau π/2.', 'D. Điện trường và từ trường biến thiên theo thời gian với cùng chu kì.']"
8,"Câu 10. Một khung dây dẫn phẳng quay đều với tốc độ góc quanh một trục cố định nằm trong mặt phẳng khung dây, trong một từ trường đều có vecto cảm ứng từ vuông góc với trục quay của khung. Suất điện động cảm ứng trong khung có biểu thức: (V). Tại thời diểm , vecto pháp tuyến của mặt phẳng khung dây hợp với vecto cảm ứng từ một góc",['A. B. C. D.']
9,Câu 11. Một máy biến áp có cuộn sơ cấp gồm 3300 vòng dây. Mắc cuộn sơ cấp vào mạng điện xoay chiều có hiệu điện thế hiệu dụng 220 V thì ở hai đầu cuộn thứ cấp để hở có một hiệu điện thế hiệu dụng 12 V. Bỏ qua hao phí của máy biến áp. Số vòng dây của cuộn thứ cấp bằng,['A. 360 vòng.B. 180 vòng.C. 120 vòng.D. 90 vòng.']


In [6]:
import pandas as pd
import re
import os

def classify_docx(json_path="./extracted/07_ThptCbq_ThptTdn-PhongBaoBui.selective.json"):
    data = pd.read_json(json_path)

    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    classified = []
    for idx, row in data.iterrows():
        text = row["text"]
        try:
            pred = clf(text)[0]
            label = label_mapping.get(pred["label"], pred["label"])
        except:
            label = "error"

        # Lưu FULL metadata
        row_data = row.to_dict()
        row_data["label"] = label

        classified.append(row_data)

    # --------- GROUP QUESTION → ANSWERS ------------
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        # Nếu là câu hỏi mới
        if label == "question":
            # Lưu question trước đó
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answers": current_answers
                })

            # Bắt đầu câu hỏi mới (giữ full metadata)
            current_question = item
            current_answers = []

        elif label == "answer":
            # Nếu chưa có prefix A/B/C thì thêm vào
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                item["text"] = prefix + text

            current_answers.append(item)

    # Lưu block cuối
    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answers": current_answers
        })

    return grouped


In [7]:
res = classify_docx("./extracted/07_ThptCbq_ThptTdn-PhongBaoBui.selective.json")  # 🔹 đường dẫn tới file JSON trích xuất

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [12]:
import json

def save_grouped_to_json(grouped, output_path="grouped_output.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(grouped, f, ensure_ascii=False, indent=4)

    print(f"Đã lưu JSON tại: {output_path}")


In [13]:
save_grouped_to_json(res, output_path="class.json")

Đã lưu JSON tại: class.json
